In [ ]:
eval "$(~/miniconda3/bin/conda shell.bash hook)"
conda activate scenicplus

# SCENIC+

In [ ]:
import os
os.chdir(path_to_wd)
import pandas as pd
import scanpy as sc
import numpy as np
import anndata as ad
from pycisTopic.cistopic_class import create_cistopic_object
from pycisTopic.lda_models import run_cgs_models_mallet
from pycisTopic.lda_models import evaluate_models
from pycisTopic.topic_binarization import binarize_topics
from pycisTopic.topic_qc import compute_topic_metrics, plot_topic_qc, topic_annotation
from pycisTopic.utils import fig2img
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)
from pycisTopic.utils import region_names_to_coordinates
import matplotlib.pyplot as plt
import pycats
from pandas.api.types import CategoricalDtype
import pickle

os.environ['MALLET_MEMORY'] = '200G'

In [ ]:
rna_meta_ad = sc.read(f"Reproducibility/Results/SEACells/TREKKER/UC_TREKKER_Malignant_rna_meta_ad.h5ad")
atac_meta_ad = sc.read(f"Reproducibility/Results/SEACells/TREKKER/UC_TREKKER_Malignant_atac_meta_ad.h5ad")

In [ ]:
rna_meta_ad = rna_meta_ad[rna_meta_ad.obs['clone3'].isin(['P03_clone_1'])==False]
atac_meta_ad = atac_meta_ad[atac_meta_ad.obs['clone3'].isin(['P03_clone_1'])==False]

## Preprocess scATAC-seq data using pycisTopic

### Creating a cisTopic object

In [ ]:
# 1. Parse peak coordinates from atac_meta_ad.var.index
atac_meta_ad.var.index = atac_meta_ad.var.index.str.replace('-', ':', n=1)
peak_index = atac_meta_ad.var.index
peak_coords = peak_index.to_series().str.extract(r'(?P<chr>[^:]+):(?P<start>\d+)-(?P<end>\d+)')
peak_coords['start'] = peak_coords['start'].astype(int)
peak_coords['end'] = peak_coords['end'].astype(int)
peak_coords.index = peak_index 

In [ ]:
# 2. Combine parsed coordinates with atac_meta_ad.var (GC_bin, etc.)
peak_metadata = pd.concat([peak_coords, atac_meta_ad.var], axis=1)

In [ ]:
# 3. Extract count matrix (ensure it's dense)
fragment_matrix = pd.DataFrame(
    data=atac_meta_ad.X.toarray().T,
    columns=atac_meta_ad.obs_names,
    index=atac_meta_ad.var_names
)

In [ ]:
# 4. Create the CistopicObject
path_to_blacklist="Reproducibility/Data/scenicplus/hg38-blacklist.v2.bed"
cistopic_obj = create_cistopic_object(
    fragment_matrix=fragment_matrix,
    path_to_blacklist=path_to_blacklist  # e.g., hg38 blacklist BED file
)

In [ ]:
# 5. Attach region and cell metadata
cistopic_obj.region_metadata = peak_metadata
cistopic_obj.cell_data = atac_meta_ad.obs

In [ ]:
out_dir = f"Reproducibility/Results/scenicplus/TREKKER/outs/"
os.makedirs(out_dir, exist_ok = True)

data_dir = f"Reproducibility/Results/scenicplus/TREKKER/data/"
os.makedirs(data_dir, exist_ok = True)

pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

### Run models

In [ ]:
# Configure path Mallet
mallet_path="~/tool/scenicplus/Mallet-202108/bin/mallet"

# Run models
models=run_cgs_models_mallet(
    cistopic_obj,
    n_topics=[2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
    n_cpu=12,
    n_iter=500,
    random_state=555,
    alpha=50,
    alpha_by_topic=True,
    eta=0.1,
    eta_by_topic=False,
    tmp_path='~/ray/tmp',
    save_path='~/ray/tmp',
    mallet_path=mallet_path,
)

pickle.dump(
    models,
    open(os.path.join(out_dir, "models.pkl"), "wb")
)

In [ ]:
# select the topic
model = evaluate_models(
    models,
    select_model = 25,
    return_model = True
)

# save
cistopic_obj.add_LDA_model(model)

pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

### Topic binarization 

In [ ]:
region_bin_topics_top_3k = binarize_topics(
    cistopic_obj, method='ntop', ntop = 3_000,
    plot=True, num_columns=5
)

region_bin_topics_otsu = binarize_topics(
    cistopic_obj, method='otsu',
    plot=True, num_columns=5
)

binarized_cell_topic = binarize_topics(
    cistopic_obj,
    target='cell',
    method='li',
    plot=True,
    num_columns=5, nbins=100)

In [ ]:
topic_qc_metrics = compute_topic_metrics(cistopic_obj)

fig_dict={}
fig_dict['CoherenceVSAssignments']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Log10_Assignments', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['AssignmentsVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Log10_Assignments', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSRegions_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Regions_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSMarginal_dist']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Marginal_topic_dist', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSGini_index']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Gini_index', var_color='Gini_index', plot=False, return_fig=True)

topic_annot = topic_annotation(
    cistopic_obj,
    annot_var='clone3',
    binarized_cell_topic=binarized_cell_topic,
    general_topic_thr = 0.2
)

### Differentially Accessible Regions

In [ ]:
imputed_acc_obj = impute_accessibility(
    cistopic_obj,
    selected_cells=None,
    selected_regions=None,
    scale_factor=10**6
)

normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

variable_regions = find_highly_variable_features(
    normalized_imputed_acc_obj,
    min_disp = 0.05,
    min_mean = 0.0125,
    max_mean = 3,
    max_disp = np.inf,
    n_bins=20,
    n_top_features=None,
    plot=True
)

len(variable_regions)

In [ ]:
# Sync cell_data index with full cell names
cistopic_obj.cell_data.index = cistopic_obj.cell_names

markers_dict= find_diff_features(
    cistopic_obj,
    imputed_acc_obj,
    variable='clone3',
    var_features=variable_regions,
    contrasts=None,
    adjpval_thr=0.05,
    log2fc_thr=np.log2(0.5),
    n_cpu=5,
    _temp_dir='~/ray/tmp/',
    split_pattern = '-'
)

print("Number of DARs found:")
print("---------------------")
for x in markers_dict:
    print(f"  {x}: {len(markers_dict[x])}")

In [ ]:
# Save region sets
os.makedirs(os.path.join(out_dir, "region_sets"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_otsu"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_top_3k"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "DARs_cell_type"), exist_ok = True)

In [ ]:
for topic in region_bin_topics_otsu:
    region_names_to_coordinates(
        region_bin_topics_otsu[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "Topics_otsu", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
for topic in region_bin_topics_top_3k:
    region_names_to_coordinates(
        region_bin_topics_top_3k[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "Topics_top_3k", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
for cell_type in markers_dict:
    region_names_to_coordinates(
        markers_dict[cell_type].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "DARs_cell_type", f"{cell_type}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
with open(f"{out_dir}consensus_regions.bed", "w") as f:
    for region in cistopic_obj.region_names:
        chrom, start, end = region.replace(":", "-").split("-")
        f.write(f"{chrom}\t{start}\t{end}\n")

pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

In [ ]:
rna_meta_ad.obs_names = rna_meta_ad.obs_names + '___cisTopic'
rna_meta_ad.raw = rna_meta_ad
rna_meta_ad.write(f'Reproducibility/Results/scenicplus/TREKKER/data/rna_meta_ad.h5ad')

## Generate a custom cisTarget database

In [ ]:
%%bash
set -euo pipefail

module use  /usr/local/package/modulefiles
module load bedtools/2.31.1

HOME_DIR="${HOME}"
WD_DIR="${path_to_wd}"
SCENICPLUS_DIR="${HOME_DIR}/tool/scenicplus"
CONDA_ENV="${HOME_DIR}/miniconda3/envs/scenicplus"

SUBSET="TREKKER"

# ==============================
# Reference & resource files
# ==============================
GENOME_FASTA="${HOME_DIR}/reference/hg38/hg38.fa"
CHROMSIZES="${SCENICPLUS_DIR}/hg38.chrom.sizes"
CBDIR="${SCENICPLUS_DIR}/aertslab_motif_colleciton/v10nr_clust_public/singletons"
MOTIF_LIST="${SCENICPLUS_DIR}/motifs.txt"

# ==============================
# Project-specific paths
# ==============================
BASE="${WD_DIR}/Reproducibility/Results/scenicplus"
OUT_DIR="${BASE}/${SUBSET}/outs2"

REGION_BED="${OUT_DIR}/consensus_regions.bed"
FASTA_FILE="${OUT_DIR}/hg38.UC_${SUBSET}.with_1kb_bg_padding.fa"
DATABASE_PREFIX="UC_${SUBSET}_1kb_bg_with_mask"

mkdir -p "${OUT_DIR}"

# ==============================
# Generate FASTA with background
# ==============================
chmod +x "${SCENICPLUS_DIR}/create_fasta_with_padded_bg_from_bed.sh"
"${SCENICPLUS_DIR}/create_fasta_with_padded_bg_from_bed.sh" \
  "${GENOME_FASTA}" \
  "${CHROMSIZES}" \
  "${REGION_BED}" \
  "${FASTA_FILE}" \
  1000 \
  yes

# ==============================
# Create motif list
# ==============================
ls "${CBDIR}" > "${MOTIF_LIST}"

# ==============================
# Environment variables
# ==============================
export PYTHONPATH="${SCENICPLUS_DIR}/create_cisTarget_databases/"
export LD_LIBRARY_PATH="${CONDA_ENV}/bin/aocl-libm/amd-libm-aocc/lib:${LD_LIBRARY_PATH:-}"

# ==============================
# Build cisTarget database
# ==============================
"${CONDA_ENV}/bin/python" \
  "${SCENICPLUS_DIR}/create_cistarget_motif_databases.py" \
  -f "${FASTA_FILE}" \
  -M "${CBDIR}" \
  -m "${MOTIF_LIST}" \
  -o "${OUT_DIR}/${DATABASE_PREFIX}" \
  --bgpadding 1000 \
  -t 20

# ==============================
# Initialize Snakemake
# ==============================
SUBSET_DIR="${BASE}/${SUBSET}"
cd "${SUBSET_DIR}"
mkdir -p scplus_pipeline outs
scenicplus init_snakemake --out_dir scplus_pipeline

In [ ]:
## Manually fill in the .../scplus_pipeline/Snakemake/config/config.yaml

In [ ]:
%%bash
set -euo pipefail

# Run Snakemake
SUBSET_DIR="${BASE}/${SUBSET}"
SNAKE_DIR="${SUBSET_DIR}/scplus_pipeline/Snakemake"

cd "${SNAKE_DIR}"

"${CONDA_ENV}/bin/snakemake" \
  --cores 20


## eRegulon score calculation

In [ ]:
import mudata
import os
import scanpy as sc
import anndata
import pandas as pd
from scenicplus.RSS import (regulon_specificity_scores, plot_rss)
import numpy as np
import matplotlib.pyplot as plt
from scenicplus.plotting.dotplot import heatmap_dotplot

In [ ]:
os.chdir(path_to_wd)

lineage = 'TREKKER'
scplus_mdata = mudata.read(f"Reproducibility/Results/scenicplus/{lineage}/scplus_pipeline/Snakemake/scplusmdata.h5mu")

In [ ]:
eRegulon_gene_AUC = anndata.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)
eRegulon_gene_AUC.obs = scplus_mdata.obs.loc[eRegulon_gene_AUC.obs_names]
eRegulon_gene_AUC.obs_names = eRegulon_gene_AUC.obs_names.str.replace('___cisTopic', '', regex=False)
eRegulon_df = pd.DataFrame(eRegulon_gene_AUC.X, index = eRegulon_gene_AUC.obs_names, columns = eRegulon_gene_AUC.var_names)

eRegulon_df.to_csv(f"Reproducibility/Results/scenicplus/{lineage}/outs/eRegulon_df.txt", sep='\t')
eRegulon_gene_AUC.obs.to_csv(f"Reproducibility/Results/scenicplus/{lineage}/outs/eRegulon_metadata.txt", sep='\t')